In [ ]:
"""
# Model Experiments and Comparison

## Compare different models
"""

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import lightgbm as lgb
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
import time

# Load data
X = np.load('../data/X.npy')
y = np.load('../data/y.npy')

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

"""
## Model Training and Evaluation
"""

models = {
    'LightGBM': lgb.LGBMRegressor(
        n_estimators=500, 
        num_leaves=31, 
        learning_rate=0.05,
        random_state=42
    ),
    'XGBoost': xgb.XGBRegressor(
        n_estimators=500, 
        max_depth=6, 
        learning_rate=0.05,
        random_state=42
    ),
    'RandomForest': RandomForestRegressor(
        n_estimators=200, 
        max_depth=15,
        random_state=42,
        n_jobs=-1
    )
}

results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    
    start_time = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start_time
    
    # Predict
    y_pred = np.expm1(model.predict(X_test))
    y_true = np.expm1(y_test)
    
    # Metrics
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    
    results[name] = {
        'MAE': mae,
        'RMSE': rmse,
        'R2': r2,
        'Train Time (s)': train_time
    }
    
    print(f"  MAE: ${mae:,.2f}")
    print(f"  RMSE: ${rmse:,.2f}")
    print(f"  R2: {r2:.4f}")
    print(f"  Time: {train_time:.2f}s")

"""
## Results Comparison
"""

results_df = pd.DataFrame(results).T
print("\n" + "="*60)
print("MODEL COMPARISON RESULTS")
print("="*60)
print(results_df.round(2))

"""
## Hyperparameter Tuning Results
"""

# Load tuning results
import json
with open('../logs/optuna_results.json', 'r') as f:
    tuning_results = json.load(f)

# Plot tuning history
import matplotlib.pyplot as plt

trials = tuning_results['trials']
values = [t['value'] for t in trials]

plt.figure(figsize=(10, 5))
plt.plot(values)
plt.xlabel('Trial')
plt.ylabel('MAE')
plt.title('Hyperparameter Tuning Progress')
plt.grid(True, alpha=0.3)
plt.show()

print(f"\nBest parameters:")
for param, value in tuning_results['best_params'].items():
    print(f"  {param}: {value}")